# So sánh Titans và TTT làm memory adapter trên Qwen2.5-0.5B (tiếng Việt)

Notebook này nạp **hai checkpoint đã train bằng hai notebook trước** (`titans_vi_memory.pt`, `ttt_linear_vi_memory.pt`; nếu có thêm `ttt_mlp_vi_memory.pt` thì so sánh luôn cả ba) và chạy **cùng một bộ kiểm tra** trên cả hai.

**Điều kiện công bằng (giống nhau):** backbone Qwen2.5-0.5B đóng băng, vị trí gắn (giữa mạng), dữ liệu và thứ tự train, kích thước memory (4 head × 64), block/chunk = 16, cách đọc ra (`out`, `gate`), optimizer, LR, số bước, tập đánh giá.

**Khác biệt cố hữu (đây chính là thứ được so sánh):**

| | Titans (NeuralMemory) | TTT (Linear / MLP) |
|---|---|---|
| Loss trong inner loop | `‖M(k) − v‖²` | `½‖k + LN(f_res(k;W)) − v‖²` |
| Cập nhật | surprise có **momentum** + **forgetting gate** (α) | một bước gradient mini-batch, **không** momentum, **không** forgetting |
| Hidden state | MLP sâu (2 lớp, ẩn ×4) | Linear hoặc MLP 2 lớp |
| Cổng học được | θ, η, α theo token | η(x) theo token |

**Lưu ý trung thực:** cả hai là *adapter trên backbone đóng băng*, không phải mô hình end-to-end như trong paper; cả hai module do thư viện/cài đặt riêng, quy mô nhỏ và train ngắn. Kết quả là của thí nghiệm này, **không** phải kết luận về hai paper ở quy mô lớn.

**Cách chạy:** thêm các file `*_vi_memory.pt` (và `data_vi.pt`) làm Input (Add Input → Notebook Output), bật GPU + Internet, Run All. `DEBUG=True` chạy thử bằng model ngẫu nhiên nhỏ.

In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "titans-pytorch==0.6.0", "transformers", "datasets", "accelerate", "matplotlib"])

In [ ]:
import os, time, math, json, re, random, glob, html as _html
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F

DEBUG = False
KINDS = ["titans", "ttt_linear", "ttt_mlp"]       # loại nào có checkpoint thì được so sánh
N_TRIALS = 12                                     # số sự kiện tự bịa mỗi khoảng cách
DISTANCES = [0, 1, 2, 4, 7]                       # số segment nhiễu chèn giữa sự kiện và câu hỏi
SEED = 0

CKPT_PATHS = {}
for _k in KINDS:
    _env = os.environ.get(f"CKPT_{_k.upper()}")
    _c = [_env] if _env else sorted(glob.glob(f"/kaggle/input/**/{_k}_vi_memory.pt", recursive=True)) + \
                             [f"/kaggle/working/ckpt_{_k}/{_k}_vi_memory.pt"]
    _c = [p for p in _c if p and os.path.exists(p)]
    if _c:
        CKPT_PATHS[_k] = _c[0]
print("Checkpoint tìm thấy:", CKPT_PATHS)
assert {"titans", "ttt_linear"} <= CKPT_PATHS.keys(), "Cần ít nhất hai checkpoint để so sánh (xem hướng dẫn ở đầu notebook)."

_blob0 = torch.load(next(iter(CKPT_PATHS.values())), map_location="cpu")
_c0 = _blob0["cfg"]
MODEL_NAME, SEG_LEN, N_SEGS = _c0["model_name"], _c0["seg_len"], _c0["n_segs"]
MEM_CHUNK, MEM_HEADS, MEM_DIM_HEAD, MEM_DEPTH = _c0["mem_chunk"], _c0["mem_heads"], _c0["mem_dim_head"], _c0["mem_depth"]
USE_MOMENTUM = _c0.get("use_momentum", True)
TITANS_INNER_LR_CAP = 1.0  # Default in older Titans checkpoints
HOOK_LAYER = None
for _k, _p in CKPT_PATHS.items():
    _c = torch.load(_p, map_location="cpu")["cfg"]
    if _k == "titans":
        TITANS_INNER_LR_CAP = _c.get("titans_inner_lr_cap", 1.0)
    for key in ["model_name", "seg_len", "n_segs", "mem_chunk", "mem_heads", "mem_dim_head", "layer"]:
        assert _c.get(key) == _c0.get(key), f"Cấu hình khác nhau giữa các checkpoint ở '{key}': {_c.get(key)} vs {_c0.get(key)}"
N_TRAIN_DOCS = int(os.environ.get("N_TRAIN_DOCS", 6000))
N_EVAL_DOCS = int(os.environ.get("N_EVAL_DOCS", 24))
EXCLUDE_TITLE_REGEX = r"vật lý|cơ học|nhiệt động|điện từ|lượng tử|tương đối|newton|động lực học|năng lượng"
BATCH_DOCS, TBPTT = 2, 2
DATA_PATH = os.environ.get("DATA_PATH", "/kaggle/input/titans-vs-ttt-input/data_vi.pt")

if DEBUG:
    N_TRIALS, DISTANCES = 2, [0, 1, 2]
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device, "| DEBUG:", DEBUG, "| cấu hình chung:", dict(seg=SEG_LEN, n_segs=N_SEGS, chunk=MEM_CHUNK, heads=MEM_HEADS))
try:
    from IPython.display import HTML, display
except Exception:
    HTML = lambda s: s
    display = print

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, Qwen2Config, Qwen2ForCausalLM

if DEBUG:
    torch.manual_seed(1234)     # backbone ngẫu nhiên phải giống nhau giữa các notebook DEBUG
    _cfg = Qwen2Config(vocab_size=500, hidden_size=64, intermediate_size=128, num_hidden_layers=4,
                       num_attention_heads=4, num_key_value_heads=2, max_position_embeddings=4096)
    lm = Qwen2ForCausalLM(_cfg)
    def encode(t): return [b % 500 for b in t.encode("utf-8")]
    def decode(ids): return "".join(chr(32 + (int(i) % 90)) for i in ids)
else:
    tok = AutoTokenizer.from_pretrained(MODEL_NAME)
    _dt = torch.float32  # Avoid FP16 overflow in the frozen backbone on T4
    try:
        lm = AutoModelForCausalLM.from_pretrained(MODEL_NAME, dtype=_dt)
    except TypeError:
        lm = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=_dt)
    def encode(t): return tok(t, add_special_tokens=False)["input_ids"]
    def decode(ids): return tok.decode(ids)

lm = lm.to(device).eval()
for p in lm.parameters():
    p.requires_grad_(False)
print("tham số backbone (đóng băng), triệu:", round(sum(p.numel() for p in lm.parameters()) / 1e6, 1))

## 1. Dữ liệu đánh giá

Dùng đúng tập đánh giá (các bài cuối, chưa từng train) của hai lần train. Nếu không tìm thấy `data_vi.pt` trong Input, notebook tự dựng lại từ Wikipedia với cùng seed (mất vài phút).

In [ ]:
def build_dataset():
    from datasets import load_dataset
    need = SEG_LEN * N_SEGS
    ds = load_dataset("wikimedia/wikipedia", "20231101.vi", split="train", streaming=True)
    ds = ds.shuffle(seed=SEED, buffer_size=10000)
    rx = re.compile(EXCLUDE_TITLE_REGEX, re.I)
    out, seen = [], 0
    for ex in ds:
        seen += 1
        if rx.search(ex["title"]) or len(ex["text"]) < need * 2:
            continue
        ids = encode(ex["text"])
        if len(ids) < need:
            continue
        out.append(torch.tensor(ids[:need]))
        if len(out) % 500 == 0:
            print(len(out), "tài liệu (đã duyệt", seen, "bài)")
        if len(out) >= N_TRAIN_DOCS + N_EVAL_DOCS:
            break
    return torch.stack(out)

_found = [DATA_PATH] if os.path.exists(DATA_PATH) else sorted(glob.glob("/kaggle/input/**/data_vi.pt", recursive=True))
if DEBUG:
    data = torch.randint(0, lm.config.vocab_size, (N_TRAIN_DOCS + N_EVAL_DOCS, SEG_LEN * N_SEGS))
elif _found:
    print("Dùng dữ liệu đã lưu:", _found[0])
    data = torch.load(_found[0], map_location="cpu")
    for path in _found[1:]:
        if not torch.equal(data, torch.load(path, map_location="cpu")):
            raise ValueError(f"Training notebooks used different datasets: {_found[0]} and {path}")
else:
    data = build_dataset()
    os.makedirs(os.path.dirname(DATA_PATH), exist_ok=True)
    torch.save(data, DATA_PATH)

train_docs, eval_docs = data[:-N_EVAL_DOCS], data[-N_EVAL_DOCS:]
print("train:", tuple(train_docs.shape), "| eval:", tuple(eval_docs.shape),
      "| tổng token train (triệu):", round(train_docs.numel() / 1e6, 1))

## 2. Nạp các adapter

In [ ]:
# ---- TTT layer (đã kiểm tra số học: dual form == vòng lặp từng token, sai số ~1e-15) ----
# ==== TTT layer (Sun et al., "Learning to (Learn at Test Time)") dùng làm memory phụ ====
# Cài đặt riêng theo mô tả trong paper (không phải code chính thức):
#   loss      l(W; x) = 1/2 || f(theta_K x; W) - theta_V x ||^2           (Eq. 4, hệ số 1/2 gộp vào lr)
#   f(x; W)   = x + LN(f_res(x; W)),  f_res = Linear (TTT-Linear) hoac MLP 2 lop GELU, hidden = 4 * dim (TTT-MLP)
#   mini-batch TTT (b = block): gradient cua moi token trong block tinh tai W dau block (t' = t - mod(t, b));
#       W_t = W_t' - sum_{s=t'+1..t} eta_s * grad l(W_t'; x_s)             (mini-batch GD)
#   output    z_t = f(theta_Q x_t; W_t)                                    (Eq. 5)
#   eta(x)    = eta_base * sigmoid(theta_lr . x)  (eta_base = 1 Linear, 0.1 MLP);  W_0 hoc duoc (outer loop)
# Tinh toan bang "dual form": khong tao W_t rieng cho tung token, ket qua tuong duong ve mat toan hoc.
# Ghi chu: chia eta cho head_dim la lua chon de on dinh (theo thong le cua cac ban cai dat tham khao), khong co trong paper.
import math
import torch
import torch.nn as nn
import torch.nn.functional as F


class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        return x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps) * self.weight


def ln_fwd(x, gamma, beta, eps=1e-6):
    mu = x.mean(-1, keepdim=True)
    var = x.var(-1, unbiased=False, keepdim=True)
    return gamma * (x - mu) / torch.sqrt(var + eps) + beta


def ln_l2_bwd(x, target, gamma, beta, eps=1e-6):
    # gradient theo x cua 1/2 || LN(x) - target ||^2 (dang dong kin)
    D = x.shape[-1]
    mu = x.mean(-1, keepdim=True)
    var = x.var(-1, unbiased=False, keepdim=True)
    std = torch.sqrt(var + eps)
    x_hat = (x - mu) / std
    y = gamma * x_hat + beta
    g = (y - target) * gamma
    return (D * g - g.sum(-1, keepdim=True) - x_hat * (g * x_hat).sum(-1, keepdim=True)) / (D * std)


def gelu_grad(x):
    return 0.5 * (1 + torch.erf(x / math.sqrt(2))) + x * torch.exp(-0.5 * x * x) / math.sqrt(2 * math.pi)


def _lower_eta(eta, k):
    # eta: (B, nh, k) -> E[t, s] = eta_s voi s <= t
    return torch.tril(eta.unsqueeze(-2).expand(*eta.shape[:-1], k, k))


def ttt_linear_block(st, q, k, v, eta, ln_w, ln_b):
    # q, k, v: (B, nh, kk, f); eta: (B, nh, kk); st: W1 (B,nh,f,f), b1 (B,nh,1,f)
    W, b = st["W1"], st["b1"]
    kk = k.size(-2)
    Z1 = k @ W + b
    G = ln_l2_bwd(Z1, v - k, ln_w, ln_b)
    loss_tok = ((k + ln_fwd(Z1, ln_w, ln_b) - v) ** 2).mean(-1)          # (B, nh, kk) "surprise"
    E = _lower_eta(eta, kk)
    A = torch.tril(q @ k.transpose(-1, -2))
    Zb = q @ W - (E * A) @ G + (b - E @ G)
    out = q + ln_fwd(Zb, ln_w, ln_b)
    W_new = W - (eta.unsqueeze(-1) * k).transpose(-1, -2) @ G
    b_new = b - (eta.unsqueeze(-1) * G).sum(-2, keepdim=True)
    return out, {"W1": W_new, "b1": b_new}, loss_tok


def ttt_mlp_block(st, q, k, v, eta, ln_w, ln_b):
    W1, b1, W2, b2 = st["W1"], st["b1"], st["W2"], st["b2"]
    kk = k.size(-2)
    Z1 = k @ W1 + b1
    X2 = F.gelu(Z1)
    Z2 = X2 @ W2 + b2
    G2 = ln_l2_bwd(Z2, v - k, ln_w, ln_b)
    G1 = (G2 @ W2.transpose(-1, -2)) * gelu_grad(Z1)
    loss_tok = ((k + ln_fwd(Z2, ln_w, ln_b) - v) ** 2).mean(-1)
    E = _lower_eta(eta, kk)
    A1 = torch.tril(q @ k.transpose(-1, -2))
    Z1b = q @ W1 - (E * A1) @ G1 + (b1 - E @ G1)
    X2b = F.gelu(Z1b)
    A2 = torch.tril(X2b @ X2.transpose(-1, -2))
    Z2b = X2b @ W2 - (E * A2) @ G2 + (b2 - E @ G2)
    out = q + ln_fwd(Z2b, ln_w, ln_b)
    e = eta.unsqueeze(-1)
    new = {
        "W1": W1 - (e * k).transpose(-1, -2) @ G1, "b1": b1 - (e * G1).sum(-2, keepdim=True),
        "W2": W2 - (e * X2).transpose(-1, -2) @ G2, "b2": b2 - (e * G2).sum(-2, keepdim=True),
    }
    return out, new, loss_tok


def flat_state(state):
    return torch.cat([state[n].flatten(1) for n in sorted(state)], dim=-1)


class TTTMem(nn.Module):
    """Giao dien: forward(x:(B,T,dim), state, want_trace) -> (read:(B,T,dim), new_state, info)."""

    def __init__(self, dim, heads=4, head_dim=64, block=16, kind="linear", eta_base=None):
        super().__init__()
        assert kind in ("linear", "mlp")
        self.nh, self.f, self.b, self.kind = heads, head_dim, block, kind
        self.eta_base = eta_base if eta_base is not None else (1.0 if kind == "linear" else 0.1)
        inner = heads * head_dim
        self.norm = RMSNorm(dim)
        self.to_q = nn.Linear(dim, inner, bias=False)
        self.to_k = nn.Linear(dim, inner, bias=False)
        self.to_v = nn.Linear(dim, inner, bias=False)
        self.to_lr = nn.Linear(dim, heads)
        nn.init.normal_(self.to_lr.weight, std=0.02)
        nn.init.zeros_(self.to_lr.bias)
        self.merge = nn.Linear(inner, dim, bias=False)
        self.ln_w = nn.Parameter(torch.ones(heads, 1, head_dim))
        self.ln_b = nn.Parameter(torch.zeros(heads, 1, head_dim))
        f = head_dim
        if kind == "linear":
            self.W1 = nn.Parameter(torch.randn(heads, f, f) * 0.02)
            self.b1 = nn.Parameter(torch.zeros(heads, 1, f))
            self.state_names = ["W1", "b1"]
        else:
            h = 4 * f
            self.W1 = nn.Parameter(torch.randn(heads, f, h) * 0.02)
            self.b1 = nn.Parameter(torch.zeros(heads, 1, h))
            self.W2 = nn.Parameter(torch.randn(heads, h, f) * 0.02)
            self.b2 = nn.Parameter(torch.zeros(heads, 1, f))
            self.state_names = ["W1", "b1", "W2", "b2"]

    def init_state(self, B):
        return {n: getattr(self, n).unsqueeze(0).expand(B, *getattr(self, n).shape).clone() for n in self.state_names}

    def forward(self, x, state=None, want_trace=False):
        B, T, _ = x.shape
        h = self.norm(x)
        sp = lambda t: t.view(B, T, self.nh, self.f).transpose(1, 2)
        q, k, v = sp(self.to_q(h)), sp(self.to_k(h)), sp(self.to_v(h))
        eta = self.eta_base * torch.sigmoid(self.to_lr(h)).transpose(1, 2) / self.f      # (B, nh, T)
        if state is None:
            state = self.init_state(B)
        block = ttt_linear_block if self.kind == "linear" else ttt_mlp_block
        outs, losses = [], []
        snaps = [flat_state(state).detach()] if want_trace else None
        for s in range(0, T, self.b):
            e = min(s + self.b, T)
            o, state, l = block(state, q[:, :, s:e], k[:, :, s:e], v[:, :, s:e], eta[:, :, s:e], self.ln_w, self.ln_b)
            outs.append(o)
            losses.append(l)
            if want_trace:
                snaps.append(flat_state(state).detach())
        z = torch.cat(outs, 2).transpose(1, 2).reshape(B, T, -1)
        read = self.merge(z)
        info = None
        if want_trace:
            info = dict(surprise=torch.cat(losses, 2).detach().float().mean((0, 1)),
                        lr=eta.detach().float().mean((0, 1)),
                        snaps=torch.stack(snaps, dim=1))
        return read, state, info

In [ ]:
from titans_pytorch import NeuralMemory, mem_state_detach

class MemLM(nn.Module):
    # Backbone đóng băng + một module memory gắn vào hidden state ở giữa mạng.
    # Cùng cách đọc ra (out, gate) cho mọi loại memory để so sánh công bằng.
    def __init__(self, lm, kind, layer=None):
        super().__init__()
        self.lm, self.kind = lm, kind
        d = lm.config.hidden_size
        if kind == "titans":
            # batch_size=MEM_CHUNK: sau mỗi chunk memory mới được cập nhật và chunk sau tính gradient theo trọng số MỚI
            # (mini-batch gradient descent của paper). Nếu để batch_size=None, thư viện luôn tính surprise theo trọng số
            # khởi tạo nên memory không bao giờ ảnh hưởng tới surprise.
            self.mem = NeuralMemory(dim=d, chunk_size=MEM_CHUNK, batch_size=MEM_CHUNK, heads=MEM_HEADS, dim_head=MEM_DIM_HEAD,
                                    momentum=USE_MOMENTUM, default_step_transform_max_lr=TITANS_INNER_LR_CAP,
                                    default_model_kwargs=dict(depth=MEM_DEPTH, expansion_factor=4.0))
            # titans-pytorch tạo tham số khởi tạo memory theo head bằng `repeat` (view stride 0) -> Adam không ghi được.
            for i, p in enumerate(self.mem.memory_model_parameters):
                self.mem.memory_model_parameters[i] = nn.Parameter(p.detach().clone().contiguous())
        elif kind in ("ttt_linear", "ttt_mlp"):
            self.mem = TTTMem(d, MEM_HEADS, MEM_DIM_HEAD, MEM_CHUNK, "linear" if kind == "ttt_linear" else "mlp")
        else:
            raise ValueError(kind)
        self.out = nn.Linear(d, d, bias=False)
        nn.init.normal_(self.out.weight, std=0.02)
        self.gate = nn.Parameter(torch.tensor(0.5))
        self.layer = layer if layer is not None else lm.config.num_hidden_layers // 2
        self.state, self.enabled, self.trace = None, True, False
        self.calls, self.w_ref = [], None
        lm.model.layers[self.layer].register_forward_hook(self._hook)

    def reset(self):
        self.state = None

    def detach_state(self):
        if self.state is None:
            return
        if self.kind == "titans":
            self.state = mem_state_detach(self.state)
        else:
            self.state = {k: v.detach() for k, v in self.state.items()}

    def fast_params(self):
        if self.kind == "titans":
            return sum(v.numel() for k, v in self.state.weights.items() if k.startswith("model.weights"))
        return sum(v.numel() for v in self.state.values())

    def trainable_parameters(self):
        return list(self.mem.parameters()) + list(self.out.parameters()) + [self.gate]

    def _mem_call(self, h):
        if self.kind == "titans":
            r, st, sur = self.mem(h, state=self.state, return_surprises=True)
            info = None
            if self.trace:
                flat = torch.cat([v.flatten(2) for k, v in st.updates.items() if k.startswith("model.weights")], dim=-1)
                info = dict(surprise=sur[0].detach().float().mean((0, 1)), lr=sur[1].detach().float().mean((0, 1)),
                            snaps=flat.detach().float())
            return r, st, info
        return self.mem(h, state=self.state, want_trace=self.trace)

    def _hook(self, module, args, output):
        if not self.enabled:
            return output
        h = output[0] if isinstance(output, tuple) else output
        r, st, info = self._mem_call(h.float())
        self.state = st
        if self.trace and info is not None:
            with torch.no_grad():
                snaps = info["snaps"]                                   # (B', n+1, P), snaps[:, 0] = trạng thái trước lần gọi
                if self.w_ref is None:
                    self.w_ref = snaps[:, 0].clone()
                dabs = (snaps - self.w_ref[:, None]).norm(dim=-1)
                dist = (dabs / self.w_ref.norm(dim=-1, keepdim=True)).mean(0)
                step = (snaps[:, 1:] - snaps[:, :-1]).norm(dim=-1).mean(0)
                self.calls.append(dict(surprise=info["surprise"].cpu().numpy(), theta=info["lr"].cpu().numpy(),
                                       dist=dist.cpu().numpy(), dist_abs=dabs.mean(0).cpu().numpy(), step=step.cpu().numpy()))
        h2 = h + (self.gate * self.out(r)).to(h.dtype)
        return (h2,) + tuple(output[1:]) if isinstance(output, tuple) else h2

def build_model(kind):
    m = MemLM(lm, kind, HOOK_LAYER)
    m.mem.to(device); m.out.to(device); m.gate.data = m.gate.data.to(device)
    return m

In [ ]:
def doc_losses(model, ids, train=False, scaler=None, reset_each=False):
    # ids: (B, N_SEGS*SEG_LEN). Trả về list loss trung bình từng segment.
    per_seg = []
    model.reset()
    for g in range(0, N_SEGS, TBPTT):
        group_loss = 0.0
        for s in range(g, min(g + TBPTT, N_SEGS)):
            if reset_each:
                model.reset()
            seg = ids[:, s * SEG_LEN:(s + 1) * SEG_LEN]
            logits = model.lm(input_ids=seg, use_cache=False).logits
            l = F.cross_entropy(logits[:, :-1].float().reshape(-1, logits.size(-1)),
                                seg[:, 1:].reshape(-1))
            loss_value = l.item()
            if not math.isfinite(loss_value):
                raise FloatingPointError(f"Non-finite loss at segment {s}: {loss_value}")
            per_seg.append(loss_value)
            group_loss = group_loss + l / N_SEGS
        if train:
            scaler.scale(group_loss).backward()
        model.detach_state()
    return per_seg


@torch.no_grad()
def evaluate(model, docs):
    acc = {"mem": [], "reset": [], "base": []}
    for i in range(0, len(docs), BATCH_DOCS):
        ids = docs[i:i + BATCH_DOCS].to(device)
        model.enabled = True
        acc["mem"].append(doc_losses(model, ids))
        acc["reset"].append(doc_losses(model, ids, reset_each=True))
        model.enabled = False
        acc["base"].append(doc_losses(model, ids))
        model.enabled = True
    res = {k: np.mean(v, axis=0) for k, v in acc.items()}      # theo segment
    summ = {k: float(v[1:].mean()) for k, v in res.items()}     # bỏ segment đầu (chưa có quá khứ)
    return summ, res

In [ ]:
import matplotlib
if DEBUG: matplotlib.use("Agg")
import matplotlib.pyplot as plt

def checksum(module):
    return float(sum(p.detach().double().abs().sum().item() for p in module.parameters()))

@torch.no_grad()
def trace_run(model, ids):
    ids = ids.to(device)
    before = (checksum(model.lm), checksum(model.mem) + checksum(model.out))
    model.reset(); model.enabled = True; model.trace = True; model.calls, model.w_ref = [], None
    for s in range(0, len(ids), SEG_LEN):
        seg = ids[s:s + SEG_LEN][None]
        model.lm(input_ids=seg, use_cache=False)
    model.trace = False
    after = (checksum(model.lm), checksum(model.mem) + checksum(model.out))
    cat = lambda key, sl: np.concatenate([c[key][sl] for c in model.calls])
    return dict(dist=cat("dist", slice(1, None)), dist_abs=cat("dist_abs", slice(1, None)), step=cat("step", slice(None)),
                surprise=cat("surprise", slice(None)), theta=cat("theta", slice(None)),
                before=before, after=after, fast_params=model.fast_params(), n_tokens=len(ids), kind=model.kind)

def plot_trace(tr):
    cfg = lm.config
    n = tr["n_tokens"]
    xs_chunk = (np.arange(len(tr["dist"])) + 1) * MEM_CHUNK
    fig, ax = plt.subplots(2, 2, figsize=(11, 7))
    ax[0, 0].plot(xs_chunk, tr["dist"], label=f"{tr['kind']}: fast-weight của memory")
    ax[0, 0].plot(xs_chunk, np.zeros_like(xs_chunk, dtype=float), "--", label="Transformer thường: trọng số cố định")
    ax[0, 0].set_title("‖M_t − M_0‖ / ‖M_0‖ theo số token"); ax[0, 0].set_xlabel("token"); ax[0, 0].legend()
    k = max(1, MEM_CHUNK)
    sm = np.convolve(tr["surprise"], np.ones(k) / k, mode="valid")
    ax[0, 1].plot(np.arange(len(sm)) + k, sm, label="surprise (loss memory, làm mượt)")
    ax[0, 1].set_xlabel("token"); ax[0, 1].set_ylabel("surprise")
    a2 = ax[0, 1].twinx(); a2.plot(xs_chunk[:len(tr["step"])], tr["step"], color="tab:red", alpha=.6, label="‖ΔM‖ mỗi chunk")
    ax[0, 1].set_title("Surprise và độ lớn cập nhật"); ax[0, 1].legend(loc="upper left"); a2.legend(loc="upper right")
    ax[1, 0].plot(tr["theta"]); ax[1, 0].set_title("learning rate trong inner loop (θ_t / η_t)"); ax[1, 0].set_xlabel("token")
    head_dim = cfg.hidden_size // cfg.num_attention_heads
    toks = np.linspace(0, max(n, 4096), 50)
    kv_mb = 2 * cfg.num_hidden_layers * cfg.num_key_value_heads * head_dim * toks * 2 / 1e6   # fp16
    mem_mb = np.full_like(toks, tr["fast_params"] * 4 / 1e6)
    ax[1, 1].plot(toks, kv_mb, label="Transformer: KV cache (tăng theo độ dài)")
    ax[1, 1].plot(toks, mem_mb, label=f"{tr['kind']}: fast-weight memory (cố định)")
    ax[1, 1].set_title("Kích thước bộ nhớ (MB)"); ax[1, 1].set_xlabel("token"); ax[1, 1].legend()
    plt.tight_layout(); plt.show()
    print(f"Checksum backbone  trước: {tr['before'][0]:.6f} | sau: {tr['after'][0]:.6f} | đổi: {tr['after'][0] != tr['before'][0]}")
    print(f"Checksum outer mem trước: {tr['before'][1]:.6f} | sau: {tr['after'][1]:.6f} | đổi: {tr['after'][1] != tr['before'][1]}")
    print(f"Fast-weight đã dịch chuyển tới {tr['dist'][-1]:.4f} lần chuẩn ban đầu (0 = không đổi).")

In [ ]:
@torch.no_grad()
def token_losses(model, ids, enabled=True):
    ids = ids.to(device)
    model.reset(); model.enabled = enabled
    losses, tids = [], []
    for s in range(0, len(ids), SEG_LEN):
        seg = ids[s:s + SEG_LEN][None]
        if seg.size(1) < 2:
            break
        logits = model.lm(input_ids=seg, use_cache=False).logits
        l = F.cross_entropy(logits[0, :-1].float(), seg[0, 1:], reduction="none")
        losses.append(l.cpu().numpy()); tids.append(seg[0, 1:].cpu().numpy())
    model.enabled = True
    return np.concatenate(losses), np.concatenate(tids)

def gain_html(model, ids, start_seg=1, max_tokens=400):
    l_mem, tids = token_losses(model, ids, True)
    l_base, _ = token_losses(model, ids, False)
    gain = (l_base - l_mem)
    lo = start_seg * (SEG_LEN - 1)
    gain, tids = gain[lo:lo + max_tokens], tids[lo:lo + max_tokens]
    mx = max(1e-6, float(np.percentile(np.abs(gain), 95)))
    parts = []
    for t, g in zip(tids, gain):
        a = min(1.0, abs(float(g)) / mx)
        col = f"rgba(0,160,0,{a:.2f})" if g > 0 else f"rgba(220,0,0,{a:.2f})"
        parts.append(f'<span style="background:{col}">{_html.escape(decode([int(t)]))}</span>')
    msg = (f"loss trung bình (đoạn hiển thị): backbone thuần {l_base[lo:lo+max_tokens].mean():.3f} | "
           f"có memory {l_mem[lo:lo+max_tokens].mean():.3f}")
    return "".join(parts), msg

def show_gain(model, ids, **kw):
    h, msg = gain_html(model, ids, **kw)
    display(HTML(h)); print(msg)

In [ ]:
@torch.no_grad()
def answer_score(model, context_ids, prompt_ids, answer_ids, mode):
    ans = torch.tensor(answer_ids, device=device)
    pr = torch.tensor(prompt_ids, device=device)
    if mode == "memory":
        model.reset(); model.enabled = True
        for s in range(0, len(context_ids), SEG_LEN):
            model.lm(input_ids=context_ids[s:s + SEG_LEN][None].to(device), use_cache=False)
        seq = torch.cat([pr, ans])
    elif mode == "window":
        model.enabled = False; model.reset()
        seq = torch.cat([pr, ans])
    else:  # "full": backbone thấy toàn bộ ngữ cảnh trong cửa sổ (cận trên)
        model.enabled = False; model.reset()
        seq = torch.cat([context_ids.to(device), pr, ans])
    n = len(answer_ids)
    logits = model.lm(input_ids=seq[None], use_cache=False, logits_to_keep=n + 1).logits[0].float()
    model.enabled = True
    logp = F.log_softmax(logits[-n - 1:-1], dim=-1)
    tok_lp = logp[torch.arange(n), ans]
    top1 = (logp.argmax(-1) == ans).all().item()
    return float(tok_lp.sum()), float(tok_lp.exp().prod()), bool(top1)

FACT   = "Theo ghi chép của phòng thí nghiệm Delta, định luật Zorvan phát biểu rằng hệ số truyền nhiệt của hợp kim Vexa bằng 7,31 đơn vị."
PROMPT = "Câu hỏi: Theo định luật Zorvan, hệ số truyền nhiệt của hợp kim Vexa bằng bao nhiêu? Trả lời: Hệ số bằng"
ANSWER = " 7,31"

In [ ]:
adapters = {}
for k, p in CKPT_PATHS.items():
    m = build_model(k)
    blob = torch.load(p, map_location=device)
    m.mem.load_state_dict(blob["mem"]); m.out.load_state_dict(blob["out"]); m.gate.data = blob["gate"].to(device)
    m.mem.eval(); m.out.eval()
    m.best_train_eval = blob.get("best"); m.train_steps = blob.get("step")
    adapters[k] = m

def use(kind):
    # chỉ bật hook của adapter được chọn (các adapter dùng chung một backbone)
    for k, m in adapters.items():
        m.enabled = (k == kind)
        m.reset()
    return adapters[kind]

PRETTY = {"titans": "Titans", "ttt_linear": "TTT-Linear", "ttt_mlp": "TTT-MLP"}
print({PRETTY[k]: f"{m.train_steps} bước, best eval loss {m.best_train_eval:.4f}" for k, m in adapters.items()})

## 3. Chi phí: tham số, kích thước fast-weight, tốc độ, bộ nhớ

In [ ]:
def timeit(fn, n=5):
    fn()
    if device == "cuda": torch.cuda.synchronize()
    t = time.time()
    for _ in range(n): fn()
    if device == "cuda": torch.cuda.synchronize()
    return (time.time() - t) / n * 1000

rows = []
seg = eval_docs[0][:SEG_LEN][None].to(device)
for k, m in adapters.items():
    use(k)
    npar = sum(p.numel() for p in m.trainable_parameters())
    with torch.no_grad():
        if device == "cuda": torch.cuda.reset_peak_memory_stats()
        ms = timeit(lambda: (m.reset(), m.lm(input_ids=seg, use_cache=False)))
        m.trace = True; m.calls, m.w_ref = [], None; m.reset(); m.lm(input_ids=seg, use_cache=False); m.trace = False
        fast = m.fast_params()
        peak = torch.cuda.max_memory_allocated() / 1e6 if device == "cuda" else float("nan")
    rows.append((PRETTY[k], npar / 1e6, fast / 1e3, ms, peak))
use(next(iter(adapters)))
for a in adapters.values(): a.enabled = False
with torch.no_grad():
    base_ms = timeit(lambda: lm(input_ids=seg, use_cache=False))
print(f"{'Mô hình':14s} | {'tham số train (M)':>18s} | {'fast-weight (nghìn)':>20s} | {'ms / segment':>12s} | {'peak MB':>8s}")
for r in rows:
    print(f"{r[0]:14s} | {r[1]:18.2f} | {r[2]:20.1f} | {r[3]:12.1f} | {r[4]:8.0f}")
print(f"{'Qwen thuần':14s} | {0:18.2f} | {0:20.1f} | {base_ms:12.1f} |")

## 4. Loss theo segment: memory có giúp không?

`reset − memory` dương nghĩa là thông tin từ các segment trước (đi qua memory) giúp dự đoán tốt hơn. Cột cuối cho thấy so với Qwen thuần (cửa sổ 1 segment).

In [ ]:
results = {}
fig, ax = plt.subplots(figsize=(8, 4))
for k, m in adapters.items():
    use(k)
    summ, per = evaluate(m, eval_docs)
    results[k] = dict(summary=summ, per_seg={a: b.tolist() for a, b in per.items()})
    ax.plot(range(1, N_SEGS + 1), per["mem"], marker="o", label=f"{PRETTY[k]}: memory mang qua segment")
    ax.plot(range(1, N_SEGS + 1), per["reset"], ls=":", marker="x", label=f"{PRETTY[k]}: xóa memory mỗi segment")
use(next(iter(adapters)))
ax.plot(range(1, N_SEGS + 1), per["base"], "k--", marker="s", label="Qwen thuần (cửa sổ 1 segment)")
ax.set_xlabel("Segment"); ax.set_ylabel("Loss"); ax.set_title("Loss theo segment trên tập đánh giá (chưa từng train)")
ax.legend(fontsize=8); plt.tight_layout(); plt.show()
print(f"{'':12s} | {'base':>8s} | {'reset':>8s} | {'memory':>8s} | {'lợi ích (reset - memory)':>26s} | {'so với Qwen thuần (base - memory)':>34s}")
for k, r in results.items():
    s = r["summary"]
    print(f"{PRETTY[k]:12s} | {s['base']:8.4f} | {s['reset']:8.4f} | {s['mem']:8.4f} | {s['reset']-s['mem']:+26.4f} | {s['base']-s['mem']:+34.4f}")

## 5. Trọng số memory thay đổi thế nào khi token đi vào

Cùng một văn bản đi vào hai memory. Backbone và tham số outer đóng băng (checksum), chỉ fast-weight đổi.

In [ ]:
traces = {}
demo_ids = eval_docs[0][: SEG_LEN * N_SEGS]
for k, m in adapters.items():
    use(k); traces[k] = trace_run(m, demo_ids)
fig, ax = plt.subplots(2, 2, figsize=(12, 7)); ax = ax.ravel()
for k, tr in traces.items():
    xs = (np.arange(len(tr["dist"])) + 1) * MEM_CHUNK
    ax[0].plot(xs, tr["dist_abs"], label=PRETTY[k])
    ax[1].plot(xs, tr["dist"], label=PRETTY[k])
    kk = MEM_CHUNK
    sm = np.convolve(tr["surprise"], np.ones(kk) / kk, mode="valid")
    ax[2].plot(np.arange(len(sm)) + kk, sm, label=PRETTY[k])
    ax[3].plot(xs[:len(tr["step"])], tr["step"], label=PRETTY[k])
for i in (0, 1):
    ax[i].plot(xs, np.zeros_like(xs, dtype=float), "k--", label="Transformer thường (cố định)")
ax[0].set_title("‖M_t − M_0‖ (tuyệt đối)"); ax[1].set_title("‖M_t − M_0‖ / ‖M_0‖ (chuẩn hóa theo trạng thái đầu)")
ax[2].set_title("surprise (loss của memory, làm mượt)"); ax[3].set_title("‖ΔM‖ mỗi chunk")
for a in ax: a.set_xlabel("token"); a.legend(fontsize=8)
plt.tight_layout(); plt.show()
for k, tr in traces.items():
    print(f"{PRETTY[k]:12s} | backbone đổi: {tr['after'][0] != tr['before'][0]} | tham số outer đổi: {tr['after'][1] != tr['before'][1]}"
          f" | fast-weight dịch chuyển {tr['dist'][-1]:.3f}x chuẩn ban đầu | kích thước fast-weight {tr['fast_params']*4/1e6:.2f} MB (cố định)")
cfg = lm.config; hd = cfg.hidden_size // cfg.num_attention_heads
print(f"KV cache của Transformer: {2*cfg.num_hidden_layers*cfg.num_key_value_heads*hd*2/1e6*1024:.1f} MB cho 1024 token, tăng tuyến tính theo độ dài.")
print("Lưu ý: hai loại memory có số chiều và thang trọng số khác nhau (W_0 của TTT rất nhỏ nên tỷ lệ chuẩn hóa dễ rất lớn), vì vậy độ lớn ‖M_t−M_0‖ giữa các loại chỉ mang tính tham khảo; điều so sánh được là liệu trọng số có thay đổi và surprise có giảm hay không.")

## 6. Token nào được hưởng lợi

Xanh = memory giúp, đỏ = memory làm tệ hơn (so với Qwen thuần cùng cửa sổ).

In [ ]:
for k, m in adapters.items():
    use(k)
    print("==", PRETTY[k])
    show_gain(m, demo_ids)

## 7. Truy xuất sự kiện tự bịa theo khoảng cách

Sự kiện (tên định luật, hợp kim, số) sinh ngẫu nhiên nên không thể nằm trong dữ liệu train của Qwen; đáp án do mình biết trước. Đo xác suất mô hình gán cho đáp án đúng khi giữa sự kiện và câu hỏi có 0, 1, 2, 4, 7 segment nhiễu. Đường *cửa sổ thường* là mức không có thông tin; *toàn bộ ngữ cảnh* là cận trên khi mọi thứ còn nằm trong cửa sổ.

In [ ]:
NAMES  = ["Zorvan", "Kelmar", "Tavris", "Brunex", "Ondira", "Veltro", "Mirkan", "Sadrel", "Yoruvi", "Pelnax"]
METALS = ["Vexa", "Lorin", "Quasa", "Dermik", "Faltor", "Nimbra", "Oskal", "Rhedia"]

def make_fact(rng):
    n, mt = rng.choice(NAMES), rng.choice(METALS)
    val = f"{rng.randint(1, 9)},{rng.randint(10, 99)}"
    fact = f"Theo ghi chép của phòng thí nghiệm Delta, định luật {n} phát biểu rằng hệ số truyền nhiệt của hợp kim {mt} bằng {val} đơn vị.\n"
    prompt = f"Câu hỏi: Theo định luật {n}, hệ số truyền nhiệt của hợp kim {mt} bằng bao nhiêu? Trả lời: Hệ số bằng"
    return fact, prompt, f" {val}"

def filler_tokens(rng, n_tokens):
    out = []
    while len(out) < n_tokens:
        d = eval_docs[rng.randrange(len(eval_docs))]
        a = rng.randrange(0, len(d) - SEG_LEN)
        out += d[a:a + SEG_LEN].tolist()
    return torch.tensor(out[:n_tokens], dtype=torch.long)

modes = [("window", "Cửa sổ thường (không thấy sự kiện)")] + [("mem:" + k, PRETTY[k]) for k in adapters] + [("full", "Toàn bộ ngữ cảnh (cận trên)")]
score = {m: {d: [] for d in DISTANCES} for m, _ in modes}
top1 = {m: {d: [] for d in DISTANCES} for m, _ in modes}
rng = random.Random(SEED)
for d in DISTANCES:
    for t in range(N_TRIALS):
        fact, prompt, ans = make_fact(rng)
        ctx = torch.cat([torch.tensor(encode(fact)), filler_tokens(rng, d * SEG_LEN)])
        p_ids, a_ids = encode(prompt), encode(ans)
        for mode, _ in modes:
            if mode.startswith("mem:"):
                m = use(mode[4:]); lp, _, ok = answer_score(m, ctx, p_ids, a_ids, "memory")
            else:
                m = use(next(iter(adapters))); lp, _, ok = answer_score(m, ctx, p_ids, a_ids, mode)
            score[mode][d].append(lp); top1[mode][d].append(ok)
    print("xong khoảng cách", d, "segment nhiễu")

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
for mode, name in modes:
    ax[0].plot(DISTANCES, [np.mean(score[mode][d]) for d in DISTANCES], marker="o", label=name)
    ax[1].plot(DISTANCES, [np.mean(top1[mode][d]) for d in DISTANCES], marker="o", label=name)
ax[0].set_title("log P(đáp án đúng), trung bình (cao hơn = tốt hơn)"); ax[1].set_title("Tỷ lệ đoán đúng toàn bộ đáp án (top-1)")
for a in ax: a.set_xlabel(f"Số segment nhiễu giữa sự kiện và câu hỏi (1 segment = {SEG_LEN} token)"); a.legend(fontsize=8)
plt.tight_layout(); plt.show()
print(f"{N_TRIALS} sự kiện tự bịa mỗi khoảng cách; đáp án là số ngẫu nhiên nên đoán mò gần như không bao giờ đúng top-1.")
print("Cột = khoảng cách:", DISTANCES)
for mode, name in modes:
    print(f"{name:38s} logP:", [round(float(np.mean(score[mode][d])), 2) for d in DISTANCES], "| top-1:", [round(float(np.mean(top1[mode][d])), 2) for d in DISTANCES])
use(next(iter(adapters)))

## 8. Tài liệu của bạn (tùy chọn)

In [ ]:
DOC_TEXT = ""      # dán một chương sách (khoảng 50-100KB)
PROMPT2  = ""      # ví dụ: "Theo giáo trình, đơn vị của đại lượng X là"
ANSWER2  = ""      # ví dụ: " kilogam"
if DOC_TEXT and PROMPT2 and ANSWER2:
    ctx = torch.tensor(encode(DOC_TEXT)); p2, a2 = encode(PROMPT2), encode(ANSWER2)
    print("Độ dài tài liệu (token):", len(ctx))
    m = use(next(iter(adapters))); lp, _, ok = answer_score(m, ctx, p2, a2, "window")
    print(f"{'Cửa sổ thường':20s} | log P(đáp án) = {lp:8.3f} | top-1: {ok}")
    for k, m in adapters.items():
        use(k); lp, _, ok = answer_score(m, ctx, p2, a2, "memory")
        print(f"{PRETTY[k]:20s} | log P(đáp án) = {lp:8.3f} | top-1: {ok}")
    for k, m in adapters.items():
        use(k); plot_trace(trace_run(m, ctx))
else:
    print("Chưa nhập DOC_TEXT/PROMPT2/ANSWER2, bỏ qua mục này.")

## 9. Lưu kết quả cho báo cáo và webapp

In [ ]:
out = dict(config=dict(seg_len=SEG_LEN, n_segs=N_SEGS, chunk=MEM_CHUNK, heads=MEM_HEADS, dim_head=MEM_DIM_HEAD, model=MODEL_NAME, debug=DEBUG),
           train_steps={kind: model.train_steps for kind, model in adapters.items()},
           loss=results,
           cost={r[0]: dict(trainable_M=r[1], fast_weight_k=r[2], ms_per_segment=r[3], peak_mb=r[4]) for r in rows},
           facts={name: dict(logp=[float(np.mean(score[m][d])) for d in DISTANCES], top1=[float(np.mean(top1[m][d])) for d in DISTANCES])
                  for m, name in modes}, distances=DISTANCES, n_trials=N_TRIALS)
path = "/kaggle/working/compare_results.json" if os.path.isdir("/kaggle/working") else os.environ.get("OUT_JSON", "compare_results.json")
json.dump(out, open(path, "w"), ensure_ascii=False, indent=1)
print("Đã lưu", path)

# Nhận xét tự động, chỉ nêu số đo, không suy diễn
ks = list(results)
best_gain = max(ks, key=lambda k: results[k]["summary"]["reset"] - results[k]["summary"]["mem"])
print("Memory cho lợi ích lớn nhất (reset − memory) trên tập đánh giá:", PRETTY[best_gain],
      {PRETTY[k]: round(results[k]['summary']['reset'] - results[k]['summary']['mem'], 4) for k in ks})
far = DISTANCES[-1]
print(f"Ở khoảng cách {far} segment, log P trung bình:", {name: round(float(np.mean(score[m][far])), 2) for m, name in modes})
print("Nếu lợi ích gần 0 hoặc đường memory không vượt đường 'cửa sổ thường', hãy báo đúng như vậy: adapter chưa học được cách dùng memory ở quy mô này.")